# TRAINING

In [2]:
#!/usr/bin/env python3
"""
QAD training script (aligned with your YOLO training config)
"""

import sys
from pathlib import Path

# Add ultralytics source to path if needed
ULTRALYTICS_SRC = Path.cwd() / "src" / "ultralytics"
if ULTRALYTICS_SRC.exists() and str(ULTRALYTICS_SRC) not in sys.path:
    sys.path.insert(0, str(ULTRALYTICS_SRC))

from ultralytics.models.yolo.detect.kd_trainer import KD_Trainer

from ultralytics import YOLO
from ultralytics.models.yolo.detect.kd_trainer import KD_Trainer

def main():
    teacher_model = YOLO(
        r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260420\20260414_2026_rider_parts_det_26m_176\weights\best.pt"
    ).model  # <-- lấy nn.Module

    config = {
        # ================= MODEL =================
        "model": "yolo26m.pt",
        # "teacher": r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260420\20260414_1230_rider_parts_det_26s_160_v3\weights\best.pt",
        "teacher": teacher_model,
        "data": r"D:\phunp\OTS\rider_parts_detection\dataset\rider_parts_det_dataset_20260420\data.yaml",

        # ================= TRAIN =================
        "epochs": 300,
        "batch": 32,
        "imgsz": 176,
        "device": 0,
        # "workers": 0,
        "exist_ok": True,
        # "fraction": 0.05,

        # giống config YOLO của bạn
        "optimizer": "auto",
        "lr0": 0.001,
        "cos_lr": True,
        "patience": 35,

        # ================= AUGMENT =================
        "hsv_h": 0.015,
        "hsv_s": 0.5,
        "hsv_v": 0.25,

        "scale": 0.1,
        "translate": 0.1,
        "degrees": 5,
        "mosaic": 0.0,
        "mixup": 0.0,
        "shear": 0,
        "perspective": 0.0,
        "fliplr": 0.5,

        # KD settings (gần với distillation_loss="cwd")
        "distillation_loss": "cwd",
        "cos_d_loss": True,
        "distill_loss_weight": 1.0,

        # ================= OUTPUT =================
        "project": r"D:\phunp\OTS\rider_parts_detection\trained_models",
        "name": r"rider_parts_det_dataset_20260420\kd_test",

        "plots": True,
        "val": True,
        "save": True,
    }

    print("=" * 60)
    print("=" * 60)
    print(f"  Model: {config['model']}")
    print(f"  Teacher: {config['teacher']}")
    print(f"  LR: {config['lr0']} | Batch: {config['batch']}")
    print("=" * 60)

    trainer = KD_Trainer(overrides=config)
    trainer.train()


if __name__ == "__main__":
    main()

  Model: yolo26m.pt
  Teacher: DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(192, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      

# RESUME

In [ ]:
import albumentations as A
from ultralytics import YOLO

custom_transforms = [
    A.ToGray(p=0.01),
    A.GaussianBlur(
        blur_limit=[3, 7],
        p=0.03,
    ),
    A.MotionBlur(
        blur_limit=[3, 7],
        allow_shifted=False,
        angle_range=[0, 0],
        direction_range=[0, 0],
        p=0.03,
    ),
    A.Downscale(
        scale_range=[0.9, 1.0],
        # interpolation_pair={"upscale":0,"downscale":0}
        p=0.02,
    ),
    A.CLAHE(p=0.02),
    A.OneOf(
        [
            A.GaussNoise(
                std_range=[0.03, 0.05],
                mean_range=[-0.05, 0.05],
                noise_scale_factor=0.5,
                # p=0.01,
            ),
            A.ISONoise(
                # p=0.01
                ),
        ],
        p=0.02,
    ),
    A.ImageCompression(quality_range=(50, 75), p=0.8),
]

model = YOLO(r"D:\phunp\OTS\lp_brand_signal_det\trained_models\lp_brand_signal_det_dataset_20260413_crop\20260414_1200_lp_brand_signal_det_26s_320\weights\last.pt")
model.train(
    resume=True,
    data=r"D:\phunp\OTS\lp_brand_signal_det\training_datasets\lp_brand_signal_det_dataset_20260413_crop\data.yaml",
    # workers=0,
    # exist_ok=True,
    # epochs=1,
    epochs=300,
    batch=16,
    imgsz=320,
    device=0,
    plots=True,
    # fraction=0.1,

    optimizer="auto",
    patience=100,
    lr0=0.001,
    multi_scale=0.25,
    cos_lr=True,

    # AUGMENTATION PARAMS
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.25,
    # hsv_h=0.0,
    # hsv_s=0.0,
    # hsv_v=0.0,

    # close_mosaic=200,
    scale=0.1,
    translate=0.1,
    degrees=3,
    mosaic=0.0,
    mixup=0.0,
    cutmix=0.0,
    concatenate=1.0,
    shear=0,
    perspective=0.0,
    fliplr=0.0,
    
    # val=False,

    project=r"D:\phunp\OTS\lp_brand_signal_det\trained_models",
    
    name=r"lp_brand_signal_det_dataset_20260413_crop\20260414_1200_lp_brand_signal_det_26s_320",
    # name=r"temp",

    min_area=0.0,
    area_thr=0.0,
    
    augmentations=custom_transforms,

    save_training_imgs=False,
    save_val=False,
    save_dir_img=r"C:\Users\Admin\Desktop\get_item",
    save_dir_lbl=r"C:\Users\Admin\Desktop\get_item",
)


New https://pypi.org/project/ultralytics/8.4.39 available  Update with 'pip install -U ultralytics'
WARNING Custom Albumentations transforms were used in the original training run but are not being restored. To preserve custom augmentations when resuming, you need to pass the 'augmentations' parameter again to get expected results. Example: 
model.train(resume=True, augmentations=[ToGray(p=0.01, method='weighted_average', num_output_channels=3), GaussianBlur(p=0.02, blur_limit=(3, 3), sigma_limit=(0.5, 3.0)), MotionBlur(p=0.01, allow_shifted=False, angle_range=(0.0, 0.0), blur_limit=(3, 3), direction_range=(0.0, 0.0)), Downscale(p=0.02, interpolation_pair={'upscale': 0, 'downscale': 0}, scale_range=(0.9, 1.0)), CLAHE(p=0.02, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8)), ImageCompression(p=0.8, compression_type='jpeg', quality_range=(50, 75))])
Ultralytics 8.4.33  Python-3.12.0 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, a

# VAL

In [1]:
from ultralytics import YOLO

conf = 0.3
imgsz = 640

train_ds = "lp_brand_signal_det_dataset_20260413_crop"

model_name = "20260414_1200_lp_brand_signal_det_26s_320"
test_ds = "helmet_PHAM_VAN_BACH_20251106_part1_1_5"

#####

model_path = fr"D:\phunp\OTS\lp_brand_signal_det\trained_models\{train_ds}\{model_name}\weights\best.pt"
src_path = r"D:\phunp\OTS\lp_brand_signal_det\test_datasets\preprocessed"

project = r"D:\phunp\OTS\lp_brand_signal_det\test_results"
name = fr"{train_ds}\20260415_1543_model_{model_name}\val\{test_ds}_conf{conf}_{imgsz}"

model = YOLO(model_path)
model.model.names = {i: str(i) for i in model.model.names} 
model.val(
    data=src_path + "\\" + test_ds + "\\data.yaml",
    project=project,
    name=name,

    conf=conf,
    imgsz=imgsz,
    batch=8,

    visualize=True,
    save_conf=True,
    device=0,
)

Ultralytics 8.4.33  Python-3.12.0 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
YOLO26s summary (fused): 122 layers, 9,467,115 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 501.3129.1 MB/s, size: 140.5 KB)
val: Scanning D:\phunp\OTS\lp_brand_signal_det\test_datasets\preprocessed\helmet_PHAM_VAN_BACH_20251106_part1_1_5\val... 1119 images, 6 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1119/1119 646.4it/s 1.7s0.1s
val: New cache created: D:\phunp\OTS\lp_brand_signal_det\test_datasets\preprocessed\helmet_PHAM_VAN_BACH_20251106_part1_1_5\val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 140/140 2.7it/s 52.4s0.5ss
                   all       1119       8856      0.809      0.705      0.779      0.586
                     0        950       1635      0.835      0.772      0.842      0.573
                     1        225        246      0.778      0.598      0.688      0.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001D5A2456C00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    

# PREDICT

In [5]:
from ultralytics import YOLO

conf = 0.3
imgsz = 176

train_ds = "rider_parts_det_dataset_20260420"

model_name = "20260421_1148_rider_parts_det_26m_176_kd_test"
test_ds = "helmet_limit"

#####

model_path = fr"D:\phunp\OTS\rider_parts_detection\trained_models\{train_ds}\{model_name}\weights\best.pt"
src_path = r"D:\phunp\OTS\rider_parts_detection\test_datasets"

project = r"D:\phunp\OTS\rider_parts_detection\test_results"
name = fr"{train_ds}\20260421_1148_model_{model_name}\predict\{test_ds}_conf{conf}_{imgsz}"

model = YOLO(model_path)
model.model.names = {i: str(i) for i in model.model.names} 
model.predict(
    source=src_path + "\\" + test_ds,
    project=project,
    name=name,
    workers=0,

    conf=conf,
    imgsz=imgsz,
    batch=32,

    # visualize=True,
    save=True,
    save_txt=True,
    save_conf=True,
    device=0,
)


WARNING imgsz=[176] must be multiple of max stride 32, updating to [192]
image 1/336 D:\phunp\OTS\rider_parts_detection\test_datasets\helmet_limit\06_22_17_375__49164__249215___0.946777__ocr_29AD40030_0.907959__origin_obj.jpeg: 192x192 3 0s, 1 1, 1 2, 7.2ms
image 2/336 D:\phunp\OTS\rider_parts_detection\test_datasets\helmet_limit\06_41_58_320__68519__266934___0.94873__ocr_29E105557_0.88988__origin_obj.jpeg: 192x192 2 0s, 1 1, 1 2, 1 3, 7.2ms
image 3/336 D:\phunp\OTS\rider_parts_detection\test_datasets\helmet_limit\07_00_38_124__5927__84352___0.930176__ocr_9991_0.743327__origin_obj.jpeg: 192x192 2 0s, 1 1, 1 2, 1 3, 7.2ms
image 4/336 D:\phunp\OTS\rider_parts_detection\test_datasets\helmet_limit\07_00_57_373__87798__170260___0.900879__ocr_99E150651_0.909885__origin_obj.jpeg: 192x192 1 0, 1 1, 1 2, 7.2ms
image 5/336 D:\phunp\OTS\rider_parts_detection\test_datasets\helmet_limit\07_01_05_970__87923__170281___0.884277__ocr_99G139915_0.91021__origin_obj.jpeg: 192x192 1 0, 1 1, 1 2, 7.2ms
ima

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: '0', 1: '1', 2: '2', 3: '3'}
 obb: None
 orig_img: array([[[123, 120, 129],
         [122, 119, 128],
         [122, 119, 128],
         ...,
         [150, 153, 158],
         [149, 152, 157],
         [148, 151, 156]],
 
        [[122, 119, 128],
         [122, 119, 128],
         [122, 119, 128],
         ...,
         [151, 154, 159],
         [149, 152, 157],
         [148, 151, 156]],
 
        [[122, 119, 128],
         [122, 119, 128],
         [122, 119, 128],
         ...,
         [151, 154, 159],
         [150, 153, 158],
         [148, 152, 157]],
 
        ...,
 
        [[103, 102, 106],
         [103, 102, 106],
         [102, 101, 105],
         ...,
         [142, 140, 146],
         [141, 139, 145],
         [140, 138, 144]],
 
        [[103, 102, 106],
         [103, 102, 106],
         [102, 101, 105],
         ...,

In [24]:
from ultralytics import YOLO

conf = 0.3
imgsz = 160

train_ds = "rider_parts_det_dataset_20260420"

model_name = "20260414_1230_rider_parts_det_26s_160_v1"
test_ds = "20260121"

#####

model_path = fr"D:\phunp\OTS\rider_parts_detection\trained_models\{train_ds}\{model_name}\weights\rider_parts_det_yolo26m_160x160_20260414_1230.onnx"
src_path = r"D:\phunp\OTS\rider_parts_detection\test_datasets"

project = r"D:\phunp\OTS\rider_parts_detection\test_results"
name = fr"{train_ds}\20260420_1414_model_{model_name}_onnx\predict\{test_ds}_conf{conf}_{imgsz}"

model = YOLO(model_path)
# model.model.names = {i: str(i) for i in model.model.names} 
model.predict(
    source=src_path + "\\" + test_ds,
    project=project,
    name=name,
    workers=0,

    conf=conf,
    imgsz=imgsz,
    batch=32,

    # visualize=True,
    save=True,
    save_txt=True,
    save_conf=True,
    device=0,
)

WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Loading D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260420\20260414_1230_rider_parts_det_26s_160_v1\weights\rider_parts_det_yolo26m_160x160_20260414_1230.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.4 with CUDAExecutionProvider

image 1/13 D:\phunp\OTS\rider_parts_detection\test_datasets\20260121\z7453491410436_609aeec5fa88351c83bf09632b214203.jpg: 160x160 2 class0s, 1 class1, 1 class2, 879.3ms
image 2/13 D:\phunp\OTS\rider_parts_detection\test_datasets\20260121\z7453491427673_a241d057a57cc1eed8bdbae1bd81163a.jpg: 160x160 2 class0s, 1 class1, 1 class2, 879.3ms
image 3/13 D:\phunp\OTS\rider_parts_detection\test_datasets\20260121\z7453491447904_98d3d35d302b4b89421e17fb4c44410d.jpg: 160x160 2 class0s, 1 class1, 1 class2, 879.3ms
image 4/13 D:\phunp\OTS\rider_parts_detection\

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'class0', 1: 'class1', 2: 'class2', 3: 'class3', 4: 'class4', 5: 'class5', 6: 'class6', 7: 'class7', 8: 'class8', 9: 'class9', 10: 'class10', 11: 'class11', 12: 'class12', 13: 'class13', 14: 'class14', 15: 'class15', 16: 'class16', 17: 'class17', 18: 'class18', 19: 'class19', 20: 'class20', 21: 'class21', 22: 'class22', 23: 'class23', 24: 'class24', 25: 'class25', 26: 'class26', 27: 'class27', 28: 'class28', 29: 'class29', 30: 'class30', 31: 'class31', 32: 'class32', 33: 'class33', 34: 'class34', 35: 'class35', 36: 'class36', 37: 'class37', 38: 'class38', 39: 'class39', 40: 'class40', 41: 'class41', 42: 'class42', 43: 'class43', 44: 'class44', 45: 'class45', 46: 'class46', 47: 'class47', 48: 'class48', 49: 'class49', 50: 'class50', 51: 'class51', 52: 'class52', 53: 'class53', 54: 'class54', 55: 'class55', 56: 'class56', 57: 'class57', 5

# OLD VERSION MODEL DETECT

In [ ]:
# %pip install --no-cache-dir "onnx" "onnxruntime-gpu"

In [ ]:
from ultralytics import YOLO

# Nạp model ONNX thay vì file .pt
model = YOLO(r"D:\phunp\OTS\lp_brand_signal_det\trained_models\lp_brand_signal_det_320_20260119.pt", task="detect") 

# Chạy predict
results = model.predict(
    source=r"D:\phunp\OTS\lp_brand_signal_det\test_datasets\helmet_PHAM_VAN_BACH_20251106\part1_col", 
    conf=0.25,
    batch=32,
    imgsz=320, 
    save=True,
    project=r"D:\phunp\OTS\lp_brand_signal_det\test_datasets\helmet_PHAM_VAN_BACH_20251106",
    name=r"part1_col_old_pred",
    )

# Duyệt kết quả
# for r in results:
#     print(r.boxes.data) # Tọa độ bounding box


Saving D:\phunp\OTS\lp_brand_signal_det\test_results\lp_brand_signal_det_dataset_20260413_crop\20260415_1543_model_20260414_1200_lp_brand_signal_det_26s_320\predict\helmet_PHAM_VAN_BACH_20251106_part1_1_5_conf0.3_320\concat_00000\stage0_Conv_features.png... (32/32)
Saving D:\phunp\OTS\lp_brand_signal_det\test_results\lp_brand_signal_det_dataset_20260413_crop\20260415_1543_model_20260414_1200_lp_brand_signal_det_26s_320\predict\helmet_PHAM_VAN_BACH_20251106_part1_1_5_conf0.3_320\concat_00000\stage1_Conv_features.png... (32/64)
Saving D:\phunp\OTS\lp_brand_signal_det\test_results\lp_brand_signal_det_dataset_20260413_crop\20260415_1543_model_20260414_1200_lp_brand_signal_det_26s_320\predict\helmet_PHAM_VAN_BACH_20251106_part1_1_5_conf0.3_320\concat_00000\stage2_C3k2_features.png... (32/128)
Saving D:\phunp\OTS\lp_brand_signal_det\test_results\lp_brand_signal_det_dataset_20260413_crop\20260415_1543_model_20260414_1200_lp_brand_signal_det_26s_320\predict\helmet_PHAM_VAN_BACH_20251106_part1

# PLOTS RESULT.PNG FROM .CSV FILE

In [4]:
from ultralytics.utils.plotting import plot_results
from pathlib import Path

# Đường dẫn tới file csv của bạn
csv_path = r"D:\phunp\OTS\people_contextual_entity_det\trained_models\pecoen_det_dataset_20260414_rs_640\20260414_1750_pecoen_det_26m_640\results.csv"

# Thực hiện vẽ
plot_results(file=csv_path)

print(f"Ảnh kết quả đã được lưu tại: {Path(csv_path).parent / 'results.png'}")


Ảnh kết quả đã được lưu tại: D:\phunp\OTS\people_contextual_entity_det\trained_models\pecoen_det_dataset_20260414_rs_640\20260414_1750_pecoen_det_26m_640\results.png


# STRIP OPTIMIZER

In [ ]:
from ultralytics.utils.torch_utils import strip_optimizer
from pathlib import Path

# 1. Đường dẫn tới file weight bạn muốn làm gọn
weight_file = r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260420\20260414_1230_rider_parts_det_26m_160_v1\weights\best.pt"

# 2. Gọi hàm strip_optimizer
# Nó sẽ tự động: chuyển sang FP16, xoá optimizer, xoá loss criterion và lưu đè lên file cũ
# strip_optimizer(weight_file)
strip_optimizer(weight_file, s=r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260420\20260414_1230_rider_parts_det_26s_160_v1\weights\rider_parts_det_yolo26m_160x160_20260414_1230.pt")

print(f"Đã hoàn tất làm gọn model: {weight_file}")


Optimizer stripped from D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260420\20260414_1230_rider_parts_det_26s_160_v1\weights\best.pt, saved as D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260420\20260414_1230_rider_parts_det_26s_160_v1\weights\rider_parts_det_yolo26s_160x160_20260414_1230.pt, 44.1MB
Đã hoàn tất làm gọn model: D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260420\20260414_1230_rider_parts_det_26s_160_v1\weights\best.pt
